# Experimento B: Impacto de la Ponderación de Clases (Cost-Sensitive Learning)

## Objetivo
Evaluar si **penalizar el error en la clase minoritaria** es suficiente para resolver el problema sin generar datos sintéticos.

## Configuración Técnica
- **Regresión Logística & Random Forest**: `class_weight='balanced'`
- **XGBoost**: `scale_pos_weight = n_negative / n_positive`
- Misma división temporal estricta que el Experimento A

## Hipótesis
El Recall debería aumentar drásticamente sin sacrificar demasiada Precisión.  
Se espera que XGBoost con ponderación supere a Random Forest.

## Comparativa
Contrastar directamente los resultados (AUPRC) contra el Experimento A.

In [ ]:
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
import xgboost as xgb

sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    COST_SENSITIVE_PARAMS, RESULTS_DIR, FIGURES_DIR, COLORS,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary, compute_class_ratio,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})
print("Configuración cargada correctamente")

---
## 1. Carga de Datos y Cálculo de Pesos

In [ ]:
transactions_df = load_transformed_data()
train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN, delta_delay=DELTA_DELAY, delta_test=DELTA_TEST,
)
print_dataset_summary(train_df, test_df, "Experimento B - Cost-Sensitive")

# Calcular scale_pos_weight para XGBoost
scale_pos_weight = compute_class_ratio(train_df[OUTPUT_FEATURE])
print(f"\nscale_pos_weight calculado: {scale_pos_weight:.2f}")

---
## 2. Entrenamiento con Ponderación de Clases

In [ ]:
def train_and_evaluate(classifier, name, train_df, test_df, input_features, output_feature):
    """Entrena y evalúa un modelo cost-sensitive."""
    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', classifier)])
    pipeline.fit(train_df[input_features], train_df[output_feature])
    
    y_pred_proba = pipeline.predict_proba(test_df[input_features])[:, 1]
    y_pred_class = (y_pred_proba >= 0.5).astype(int)
    
    result = {
        'name': name,
        'pipeline': pipeline,
        'y_pred_proba_test': y_pred_proba,
        'auc_roc': metrics.roc_auc_score(test_df[output_feature], y_pred_proba),
        'avg_precision': metrics.average_precision_score(test_df[output_feature], y_pred_proba),
        'accuracy': metrics.accuracy_score(test_df[output_feature], y_pred_class),
        'recall': metrics.recall_score(test_df[output_feature], y_pred_class),
        'precision': metrics.precision_score(test_df[output_feature], y_pred_class, zero_division=0),
        'f1': metrics.f1_score(test_df[output_feature], y_pred_class),
    }
    
    print(f"\n  {name}:")
    print(f"    AUC ROC:  {result['auc_roc']:.4f}  |  AUPRC: {result['avg_precision']:.4f}")
    print(f"    Recall:   {result['recall']:.4f}  |  Precision: {result['precision']:.4f}  |  F1: {result['f1']:.4f}")
    return result

# Configurar clasificadores cost-sensitive
xgb_params = {**COST_SENSITIVE_PARAMS["XGBoost"], "scale_pos_weight": scale_pos_weight}

classifiers_b = {
    "LR (balanced)": LogisticRegression(**COST_SENSITIVE_PARAMS["Logistic Regression"]),
    "RF (balanced)": RandomForestClassifier(**COST_SENSITIVE_PARAMS["Random Forest"]),
    "XGBoost (weighted)": xgb.XGBClassifier(**xgb_params),
}

print("=" * 60)
print("  RESULTADOS DEL EXPERIMENTO B: COST-SENSITIVE")
print("=" * 60)

results_b = {}
for name, clf in classifiers_b.items():
    results_b[name] = train_and_evaluate(clf, name, train_df, test_df, INPUT_FEATURES, OUTPUT_FEATURE)

---
## 3. Comparativa A vs B

In [ ]:
# Cargar resultados del Exp A para comparar
with open(RESULTS_DIR / 'experiment_a_predictions.pkl', 'rb') as f:
    results_a = pickle.load(f)

# Tabla comparativa A vs B
comparison_data = []
for exp_label, results_dict in [("A (Baseline)", results_a), ("B (Cost-Sensitive)", results_b)]:
    for name, res in results_dict.items():
        comparison_data.append({
            'Experimento': exp_label,
            'Modelo': res.get('name', name),
            'AUC ROC': res['auc_roc'],
            'AUPRC': res['avg_precision'],
            'Recall': res['recall'],
            'Precision': res['precision'],
        })

comparison_table = pd.DataFrame(comparison_data).round(4)
print("\nComparativa Experimento A vs B:")
print("=" * 80)
display(comparison_table)

# Guardar resultados
results_table_b = pd.DataFrame({
    'Modelo': [r['name'] for r in results_b.values()],
    'AUC ROC': [r['auc_roc'] for r in results_b.values()],
    'AUPRC': [r['avg_precision'] for r in results_b.values()],
    'Recall': [r['recall'] for r in results_b.values()],
    'Precision': [r['precision'] for r in results_b.values()],
    'F1-Score': [r['f1'] for r in results_b.values()],
}).set_index('Modelo').round(4)

results_table_b.to_csv(RESULTS_DIR / 'experiment_b_results.csv')
comparison_table.to_csv(RESULTS_DIR / 'experiment_a_vs_b_comparison.csv', index=False)

# Guardar predicciones
results_b_save = {name: {k: v for k, v in res.items() if k != 'pipeline'} for name, res in results_b.items()}
with open(RESULTS_DIR / 'experiment_b_predictions.pkl', 'wb') as f:
    pickle.dump(results_b_save, f)

print("\n✓ Resultados del Experimento B guardados")

---
## 4. Visualización Comparativa A vs B

# Preparar datos para la comparación visual
# Mapeo de modelos: A (baseline) → B (cost-sensitive)
model_pairs = [
    ("Logistic Regression", "LR (balanced)"),
    ("Random Forest", "RF (balanced)"),
    ("XGBoost", "XGBoost (weighted)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Gráfico 1: AUPRC comparación A vs B ---
ax = axes[0]
x_pos = np.arange(len(model_pairs))
width = 0.35
auprc_a = [results_a[a]['avg_precision'] for a, b in model_pairs]
auprc_b = [results_b[b]['avg_precision'] for a, b in model_pairs]
bars1 = ax.bar(x_pos - width/2, auprc_a, width, label='A (Baseline)', color=COLORS['baseline'])
bars2 = ax.bar(x_pos + width/2, auprc_b, width, label='B (Cost-Sensitive)', color=COLORS['cost_sensitive'])
ax.set_ylabel('AUPRC', fontsize=12)
ax.set_title('AUPRC: Baseline vs Cost-Sensitive', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels([a for a, b in model_pairs], rotation=15, ha='right')
ax.legend()
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)

# --- Gráfico 2: Recall comparación A vs B ---
ax = axes[1]
recall_a = [results_a[a]['recall'] for a, b in model_pairs]
recall_b = [results_b[b]['recall'] for a, b in model_pairs]
bars1 = ax.bar(x_pos - width/2, recall_a, width, label='A (Baseline)', color=COLORS['baseline'])
bars2 = ax.bar(x_pos + width/2, recall_b, width, label='B (Cost-Sensitive)', color=COLORS['cost_sensitive'])
ax.set_ylabel('Recall (Fraude)', fontsize=12)
ax.set_title('Recall: Baseline vs Cost-Sensitive', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels([a for a, b in model_pairs], rotation=15, ha='right')
ax.legend()
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)

# --- Gráfico 3: Curvas PR comparadas (mejor modelo A vs mejor modelo B) ---
ax = axes[2]
# Curvas PR del Exp B
for name, res in results_b.items():
    prec, rec, _ = metrics.precision_recall_curve(test_df[OUTPUT_FEATURE], res['y_pred_proba_test'])
    ax.plot(rec, prec, linewidth=2, label=f"B: {name} (AP={res['avg_precision']:.3f})")

# Curvas PR del Exp A (cargadas desde pkl, contienen y_pred_proba_test)
for name, res in results_a.items():
    prec, rec, _ = metrics.precision_recall_curve(test_df[OUTPUT_FEATURE], res['y_pred_proba_test'])
    ax.plot(rec, prec, linewidth=1, linestyle='--', alpha=0.6,
            label=f"A: {name} (AP={res['avg_precision']:.3f})")

baseline_fraud_rate = test_df[OUTPUT_FEATURE].mean()
ax.axhline(y=baseline_fraud_rate, color='gray', linestyle=':', alpha=0.5,
           label=f'Aleatorio ({baseline_fraud_rate:.4f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curvas PR: A (Baseline) vs B (Cost-Sensitive)', fontsize=14)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim([0, 1.01]); ax.set_ylim([0, 1.01])

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_b_comparison_a_vs_b.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFigura guardada en: {FIGURES_DIR / 'experiment_b_comparison_a_vs_b.png'}")

---
## 5. Conclusiones del Experimento B

**Resultado esperado:** La ponderación de clases mejora significativamente el **Recall** sin necesidad de datos sintéticos.

- Los modelos cost-sensitive deberían capturar más fraudes (mayor Recall) que los modelos baseline.
- El trade-off Precision-Recall se visualiza claramente en las curvas PR.
- XGBoost con `scale_pos_weight` debería ofrecer el mejor equilibrio entre detección y falsos positivos.